> **MIT License** - Copyright (c) 2026 QuantWall. Full text: https://opensource.org/license/mit
>
> **Disclaimer.** This content is general information and software-engineering education only. It is not financial product advice, a recommendation, or an offer to buy or sell any financial product, and it does not take into account your objectives, financial situation or needs. The author does not hold an Australian Financial Services Licence and is not a registered Commodity Trading Advisor. Trading futures and leveraged products carries a high risk of loss. Consult a licensed adviser before making any financial decision.
>
> **HYPOTHETICAL OR SIMULATED PERFORMANCE RESULTS HAVE CERTAIN LIMITATIONS. UNLIKE AN ACTUAL PERFORMANCE RECORD, SIMULATED RESULTS DO NOT REPRESENT ACTUAL TRADING. ALSO, SINCE THE TRADES HAVE NOT BEEN EXECUTED, THE RESULTS MAY HAVE UNDER-OR-OVER COMPENSATED FOR THE IMPACT, IF ANY, OF CERTAIN MARKET FACTORS, SUCH AS LACK OF LIQUIDITY. SIMULATED TRADING PROGRAMS IN GENERAL ARE ALSO SUBJECT TO THE FACT THAT THEY ARE DESIGNED WITH THE BENEFIT OF HINDSIGHT. NO REPRESENTATION IS BEING MADE THAT ANY ACCOUNT WILL OR IS LIKELY TO ACHIEVE PROFIT OR LOSSES SIMILAR TO THOSE SHOWN.**
>
> All data shown is synthetic or publicly available sample data. Features are standard textbook baselines (lagged returns, ATR, RSI, moving averages, volume). No proprietary signal is disclosed or implied.

# What you need: every number in the video

The compute times come from one measured workload; the prices are rounded list prices seen in October 2026 (no provider names; they change, so check before you spend). Requirements: `numpy`. The measurement cell takes a few seconds.

In [ ]:
MEASURED_S = 2.3                 # seconds per simulated world on one core (median of 5 runs, 2026-10-31)
WORLDS = 2 * 6000 * 4            # noise worlds + planted worlds, 6,000 each, for 4 test designs = 48,000

LAPTOP_CORES = 8
RENTED_CORES = 32
RENT_ON_DEMAND = (1.00, 1.65)    # $ per hour, 32 vCPUs, on demand (third-party price listings, range across providers)
SETUP_MIN = 10                   # minutes to start the machine, copy the code in and the results out

DATA_ENTRY_MONTH = 30            # $ a month, end-of-day futures data, entry tier (one provider's list price)


def core_hours():
    return MEASURED_S * WORLDS / 3600


def run_hours(cores):
    return core_hours() / cores


def rented_cost(prices):
    hours = run_hours(RENTED_CORES) + SETUP_MIN / 60
    return tuple(p * hours for p in prices)


## 1. The workload, measured on your machine
One heavy run of one test = 48,000 simulated worlds (6,000 noise + 6,000 planted, for 4 test designs). Time one world here.

In [ ]:
import time
import numpy as np


def one_world(rng, markets=40, days=7560, nulls=1000, look=252):
    """One simulated world: 40 markets, 30 years of daily returns, a 12-month trend signal decided the day
    before, and a random-timing null with `nulls` shuffles (each market's signal shifted at random)."""
    r = rng.normal(0, 0.01, (days, markets))
    c = np.cumsum(r, axis=0)
    sig = np.sign(c[look:] - c[:-look])[:-1]
    fwd = r[look + 1:]
    stat = (sig * fwd).mean()
    beat = 0
    for _ in range(nulls):
        k = rng.integers(1, sig.shape[0], markets)
        null = np.stack([np.roll(sig[:, j], k[j]) for j in range(markets)], axis=1)
        beat += (null * fwd).mean() >= stat
    return beat / nulls


rng = np.random.default_rng(1)
nulls = 50                                   # a short measurement; scaled to 1,000 shuffles below
t0 = time.perf_counter()
one_world(rng, nulls=nulls)
my_seconds = (time.perf_counter() - t0) * 1000 / nulls
print(f"one world on one core of this machine: {my_seconds:.1f} s   (video: {MEASURED_S} s)")

## 2. One heavy run, your machine or a rented one
Ideal scaling by core count (real runs are a little slower). Swap in your own measurement to see your numbers.

In [ ]:
for name, sec in (("video", MEASURED_S), ("this machine", my_seconds)):
    total_h = sec * WORLDS / 3600
    print(f"{name}: {total_h:.1f} core-hours | 8 cores {total_h / 8:.1f} h | 32 cores {total_h / 32 * 60:.0f} min")

## 3. What renting costs
The run on 32 cores plus 10 minutes to start, copy in and copy out, at on-demand prices.

In [ ]:
lo, hi = rented_cost(RENT_ON_DEMAND)
print(f"one heavy run, rented: ${lo:.2f}-{hi:.2f}")

## 4. How big the data is
Daily prices for 40 markets over 30 years: open, high, low, close and volume as 8-byte numbers.

In [ ]:
print(f"{40 * 7560 * 5 * 8 / 1e6:.1f} MB")

### Try this
- Halve `nulls` in `one_world`: how does the run time change? Is the test still trustworthy with fewer shuffles?
- Put your own laptop's core count in section 2. How long would a heavy run take on your machine?

*Education only. Not financial advice.*